CaptchaSolve30k — human-only detection vs supervised detection against an UNSEEN replay attacker

Follow-up to `CaptchaSolve30k_RF_vs_GSFusion_5.ipynb`. Humans, bots (L1–L4 synthetic, L5 replay re-sampled as one device), features and the per-seed splits are **identical** to v5 (same streaming order, same seeds), so the E2 numbers are directly comparable.

**Question.** In E2 every supervised model trained on L1–L4 loses at least half its replay recall (RF 0.46, GS-Fusion 0.25). A detector that learns only what *humans* look like cannot overfit to known bots. Does it catch unseen replay — alone, or combined with RF?

**Detectors (all trained without any bot)**
- on the 26 stat features: Isolation Forest, Gaussian mixture, Mahalanobis (Ledoit–Wolf), kNN distance;
- **SSL encoder:** a GRU pretrained on *other* CaptchaSolve30k humans (disjoint sessions streamed after the main ones) to predict the next movement step (dx, dy, button) at 30/60 Hz. Scores: mean next-step prediction error, and kNN distance in its embedding space.

**Hybrids:** RF (stat, supervised on humans + L1–L4) and a human-only detector, combined with the “any view flags” rule: each has a p-value from validation humans, the session is flagged if any p ≤ 5 %/k (Bonferroni), so the total FPR budget stays 5 %.

**Pre-registered test (fixed before running).** Primary hybrid: **RF | SSL error**. It passes if (a) its replay recall (mean of `human_replay`, `human_replay_raw`) ≥ RF's + 0.10 in at least 4 of 5 seeds, and (b) its mean recall on every one of L1–L4 stays ≥ 0.95. All other hybrids and detectors are secondary/exploratory.

Run on a GPU (local 4090 or Colab T4). Results go to `./captcha_human_only`.


In [ ]:
# 1. Setup
%pip install -q datasets lightgbm
import os, math, time, random, warnings, base64, struct, copy
from collections import Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
import torch.nn.functional as F
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try: display
except NameError: display = print
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu": print("⚠️  No GPU: Runtime → Change runtime type → T4 GPU, otherwise the neural models take hours.")

In [ ]:
# 2. Configuration
N_PER_GAME    = 2000          # human sessions per game (3 games → 6,000); raise for the final run
INCLUDE_TOUCH = False         # touchscreen sessions behave differently; keep them out by default
MIN_TICKS     = 240           # at least 1 s of input
TICK          = 1 / 240       # seconds per physics tick
BOT_SHARE     = 0.25          # bot sessions per level = BOT_SHARE × usable humans (5 levels → 1.25× humans)
FAST          = True          # True: 30 Hz sequences of ≤ 256 steps, fewer epochs (≈ 4× faster); False for the final run
SEQ_STRIDE    = 8 if FAST else 4      # 240 Hz → 30 / 60 Hz for the sequence view
SEQ_MAX       = 256 if FAST else 512  # max sequence steps (≈ 8.5 s)
SEEDS         = [0, 1, 2, 3, 4]
TARGET_FPR    = 0.05
D, EPOCHS, PATIENCE, BATCH, LR = (32, 25, 4, 512, 2e-3) if FAST else (32, 40, 6, 256, 1e-3)
LAMBDA_BAL    = 0.1
REPLAY_RESAMPLE = True        # replay: rebuild the stitched path in continuous time, then re-sample it at the SOURCE human's
                              # update rate and lattice (one device per session, as a real browser replay bot would have)
MATCH_SENSOR  = True          # L1–L4 copy the source human's input update rate and coordinate lattice (replay is left as recorded)
SKIP_SEQ      = None          # None = automatic: skip the LSTM-based models when there is no GPU (they take hours on CPU)
BASE = "/content" if os.path.isdir("/content") else os.path.abspath(".")   # Colab or a local Jupyter
import datetime
RUN_DATE = datetime.date.today().isoformat()   # every result folder, file and zip gets this suffix, e.g. _2026-11-12
def date_files(folder=None):
    """Add _RUN_DATE to every result file in the folder (progress .pkl files keep their name so a run can resume)."""
    folder = folder or OUT
    for p in os.listdir(folder):
        root, ext = os.path.splitext(p)
        if ext == ".pkl" or root.endswith(RUN_DATE): continue
        os.replace(os.path.join(folder, p), os.path.join(folder, f"{root}_{RUN_DATE}{ext}"))
OUT = f"{BASE}/captcha_human_only_{RUN_DATE}"; os.makedirs(OUT, exist_ok=True)
LEVELS = ["linear", "linear_jitter", "bezier_fitts", "bezier_tremor", "human_replay", "human_replay_raw"]
REPLAY = ["human_replay", "human_replay_raw"]          # both are hidden from training in E2
GAMES = ["sheep-herding", "thread-the-needle", "polygon-stacking"]
rng = np.random.default_rng(2026); random.seed(2026)
# human-only detectors
PRETRAIN_PER_GAME = 3000          # extra human sessions per game for SSL pretraining, streamed AFTER the main ones (disjoint)
SSL_HID, SSL_EPOCHS, SSL_PATIENCE, SSL_BATCH, SSL_LR = (64, 12, 3, 256, 1e-3) if FAST else (96, 30, 4, 256, 1e-3)
KNN_K = 10
LEVELS_SYN = ["linear", "linear_jitter", "bezier_fitts", "bezier_tremor"]


In [ ]:
# 3. Load human sessions (streamed). The first N_PER_GAME per game are the SAME humans as in v5;
#    the next PRETRAIN_PER_GAME per game go to the SSL pretraining set and are never used anywhere else.
from datasets import load_dataset
def stream():
    for f in ["train.jsonl", "data.jsonl"]:
        try:
            ds = load_dataset("Capycap-AI/CaptchaSolve30k", data_files=f, split="train", streaming=True)
            next(iter(ds)); print("reading", f); return ds
        except Exception as e:
            print(f"{f}: not usable ({type(e).__name__})")
    return load_dataset("Capycap-AI/CaptchaSolve30k", split="train", streaming=True)

humans, pre, cnt, pcnt, skipped = [], [], Counter(), Counter(), Counter()
t0 = time.time()
for r in stream():
    g = r.get("gameType")
    if g not in GAMES or (cnt[g] >= N_PER_GAME and pcnt[g] >= PRETRAIN_PER_GAME):
        if all(cnt[x] >= N_PER_GAME and pcnt[x] >= PRETRAIN_PER_GAME for x in GAMES): break
        continue
    if r.get("touchscreen") and not INCLUDE_TOUCH: skipped["touch"] += 1; continue
    ti = r.get("tickInputs") or []
    if len(ti) < MIN_TICKS: skipped["short"] += 1; continue
    a = np.array([[d["x"], d["y"], float(bool(d["isDown"]))] for d in ti], dtype=np.float32)
    if cnt[g] < N_PER_GAME: humans.append(dict(game=g, xy=a)); cnt[g] += 1
    else: pre.append(a); pcnt[g] += 1
print(f"{len(humans)} human sessions {dict(cnt)} · {len(pre)} pretraining sessions {dict(pcnt)} in {time.time() - t0:.0f}s; skipped {dict(skipped)}")


In [ ]:
# 4. Task skeletons: the waypoints a session must pass through (presses, releases, turns, every 0.4 s)
def skeleton(a, every=96, turn_deg=50, step=12):
    """Indices of waypoints in a tick array a (T×3)."""
    T = len(a); keep = {0, T - 1}
    keep |= set((np.flatnonzero(np.diff(a[:, 2]) != 0) + 1).tolist())           # press / release: first tick of the new state
    keep |= set(range(0, T, every))
    p = a[::step, :2]
    if len(p) > 2:                                                               # sharp turns
        v = np.diff(p, axis=0); ang = np.arctan2(v[:, 1], v[:, 0])
        dang = np.abs((np.diff(ang) + np.pi) % (2 * np.pi) - np.pi)
        moving = (np.linalg.norm(v[:-1], axis=1) > 0.5) & (np.linalg.norm(v[1:], axis=1) > 0.5)
        keep |= set(((np.flatnonzero((dang > np.radians(turn_deg)) & moving) + 1) * step).tolist())
    return np.array(sorted(i for i in keep if i < T))

for h in humans: h["wp"] = skeleton(h["xy"])

In [ ]:
# 5. Splits of the humans: a fixed replay pool, the rest is split per seed later
idx = np.arange(len(humans))
games_arr = np.array([h["game"] for h in humans])
work_idx, replay_idx = train_test_split(idx, test_size=0.20, stratify=games_arr, random_state=7)
print(f"humans for train/val/test: {len(work_idx)} · replay pool (never trained or tested on): {len(replay_idx)}")

# replay library: every waypoint-to-waypoint segment of the pool humans
LIB, LIB_KEY = [], []
for i in replay_idx:
    a, wp = humans[i]["xy"], humans[i]["wp"]
    for s, e in zip(wp[:-1], wp[1:]):
        seg = a[s:e + 1, :2]
        d = float(np.hypot(*(seg[-1] - seg[0])))
        if d < 2 or len(seg) < 4: continue
        th = math.atan2(seg[-1, 1] - seg[0, 1], seg[-1, 0] - seg[0, 0])
        LIB.append(seg); LIB_KEY.append((d, math.cos(th), math.sin(th)))
LIB_KEY = np.array(LIB_KEY); D_MED = np.median(LIB_KEY[:, 0])
KD = cKDTree(np.c_[LIB_KEY[:, 0] / D_MED, LIB_KEY[:, 1], LIB_KEY[:, 2]])
print(f"replay library: {len(LIB):,} human movement segments")

In [ ]:
# 6. Bot generators — same waypoints and button states as the source human, own movement
def ticks_for(dur): return max(2, int(round(dur / TICK)))

def seg_linear(A, B, speed, noise=0.0):
    d = np.hypot(*(B - A)); n = ticks_for(d / speed)
    t = np.linspace(0, 1, n)[:, None]
    p = A + (B - A) * t
    return p + rng.normal(0, noise, p.shape) if noise else p

def seg_bezier(A, B, tremor=False):
    d = np.hypot(*(B - A))
    if d < 0.5: return np.repeat(A[None], 2, 0)
    dur = (0.10 + 0.11 * math.log2(d / 6 + 1)) * math.exp(rng.normal(0, 0.25))
    n = ticks_for(dur); t = np.linspace(0, 1, n)
    if tremor:                                                   # two-phase sub-movement speed profile
        t = np.where(t < 0.7, 0.85 * (t / 0.7) ** 1.2, 0.85 + 0.15 * ((t - 0.7) / 0.3))
    s = t * t * (3 - 2 * t)
    nrm = np.array([-(B - A)[1], (B - A)[0]]) / d
    bend = rng.uniform(-0.25, 0.25) * d
    C1, C2 = A + (B - A) * 0.3 + nrm * bend, A + (B - A) * 0.7 + nrm * bend * 0.5
    u = (1 - s)[:, None]; s2 = s[:, None]
    p = u**3 * A + 3 * u**2 * s2 * C1 + 3 * u * s2**2 * C2 + s2**3 * B
    if tremor:
        f = rng.uniform(8, 12); amp = rng.uniform(0.15, 0.5); ph = rng.uniform(0, 6.3, 2)
        tt = np.arange(n) * TICK
        p = p + amp * np.c_[np.sin(2 * np.pi * f * tt + ph[0]), np.sin(2 * np.pi * f * tt + ph[1])]
    return p

def seg_bezier_tremor(A, B):
    d = np.hypot(*(B - A))
    if d > 10 and rng.random() < 0.2:                            # overshoot past B, then correct back
        O = B + (B - A) / d * rng.uniform(2, 6) + rng.normal(0, 1, 2)
        return np.vstack([seg_bezier(A, O, tremor=True), seg_linear(O, B, rng.uniform(40, 90))[1:]])
    return seg_bezier(A, B, tremor=True)

def seg_replay(A, B):
    d = float(np.hypot(*(B - A)))
    if d < 2: return seg_linear(A, B, 60)
    th = math.atan2(B[1] - A[1], B[0] - A[0])
    _, cand = KD.query([d / D_MED, math.cos(th), math.sin(th)], k=8)
    seg = LIB[int(rng.choice(np.atleast_1d(cand)))]
    v0 = seg[-1] - seg[0]; sc = d / max(np.hypot(*v0), 1e-6)
    rot = th - math.atan2(v0[1], v0[0]); c, s_ = math.cos(rot), math.sin(rot)
    q = (seg - seg[0]) * sc
    return A + np.c_[q[:, 0] * c - q[:, 1] * s_, q[:, 0] * s_ + q[:, 1] * c]

def seg_replay_raw(A, B):
    """Translation only: the nearest human segment by length and direction, NOT rotated or scaled.
    It ends near B rather than on it; the next move starts from wherever it ended."""
    d = float(np.hypot(*(B - A)))
    if d < 2: return seg_linear(A, B, 60)
    th = math.atan2(B[1] - A[1], B[0] - A[0])
    _, cand = KD.query([d / D_MED, math.cos(th), math.sin(th)], k=3)
    seg = LIB[int(rng.choice(np.atleast_1d(cand)))]
    return A + (seg - seg[0])

def sensor_profile(a):
    """(update rate per 240 Hz tick while moving, smallest coordinate step) of a session."""
    p = a[:, :2]; ch = np.any(np.diff(p, axis=0) != 0, axis=1)
    near = np.convolve(ch.astype(float), np.ones(9), "same") > 0
    r = float(ch[near].mean()) if near.any() else 1.0
    d = np.abs(np.diff(p, axis=0)).ravel(); d = d[d > 1e-6]
    return min(max(r, 0.05), 1.0), (float(np.percentile(d, 1)) if len(d) else 0.0)

def apply_sensor(b, r, q):
    """Sample-and-hold at rate r (random phase), then snap to a lattice of step q. The button column is untouched."""
    if r < 0.999:
        k = np.arange(len(b)); ph = rng.random()
        upd = np.floor(k * r + ph) > np.floor((k - 1) * r + ph); upd[0] = True
        b[:, :2] = b[np.maximum.accumulate(np.where(upd, k, 0)), :2]
    if q > 1e-4: b[:, :2] = np.round(b[:, :2] / q) * q
    return b

def resample_replay(b, r, q, max_gap=12):
    """Stitched replay → one device: interpolate between position updates (gaps ≤ max_gap ticks = 50 ms;
    longer gaps are real pauses and stay held), then sample-and-hold at rate r and snap to lattice q."""
    p = b[:, :2].astype(np.float64); t = np.arange(len(b))
    ch = np.r_[True, np.any(np.diff(p, axis=0) != 0, axis=1)]; k = np.flatnonzero(ch)
    if len(k) >= 2:
        prev = k[np.searchsorted(k, t, side="right") - 1]
        nxt = k[np.minimum(np.searchsorted(k, t, side="left"), len(k) - 1)]
        smooth = np.c_[np.interp(t, k, p[k, 0]), np.interp(t, k, p[k, 1])]
        p = np.where(((nxt - prev) <= max_gap)[:, None], smooth, p[prev])
    b = b.copy(); b[:, :2] = p
    r1 = sensor_profile(apply_sensor(b.copy(), r, q))[0]                 # one correction step, as for L1–L4
    return apply_sensor(b, min(1.0, r * r / max(r1, 1e-3)), q)

def make_bot(h, level):
    a, wp = h["xy"], h["wp"]
    speed = rng.uniform(60, 140)
    out = []; cur = a[0, :2].astype(float)
    for s, e in zip(wp[:-1], wp[1:]):
        A, B, down = a[s, :2].astype(float), a[e, :2].astype(float), a[s, 2]
        if level == "human_replay_raw": A = cur
        if level == "linear":          p = seg_linear(A, B, speed)
        elif level == "linear_jitter": p = seg_linear(A, B, speed * math.exp(rng.normal(0, 0.4)), noise=0.3)
        elif level == "bezier_fitts":  p = seg_bezier(A, B)
        elif level == "bezier_tremor": p = seg_bezier_tremor(A, B)
        elif level == "human_replay":  p = seg_replay(A, B)
        else:                          p = seg_replay_raw(A, B)
        cur = p[-1].astype(float)
        out.append(np.c_[p, np.full(len(p), down)])
        if a[e, 2] != down:                                          # press or release at B: dwell, then switch
            pause = {"linear": 0.08, "linear_jitter": rng.uniform(0.02, 0.25)}.get(level, rng.lognormal(math.log(0.08), 0.6))
            n = ticks_for(pause)
            out.append(np.c_[np.repeat(p[-1:], n, 0), np.full(n, a[e, 2])])
    b = np.vstack(out).astype(np.float32)
    if MATCH_SENSOR:
        if "sensor" not in h: h["sensor"] = sensor_profile(a)
        r, q = h["sensor"]
        if level not in REPLAY:                                    # replay keeps the human update timing of its segments
            r1 = sensor_profile(apply_sensor(b.copy(), r, q))[0]   # one correction step: lattice snapping drops updates
            r = min(1.0, r * r / max(r1, 1e-3))
            b = apply_sensor(b, r, q)
        elif REPLAY_RESAMPLE: b = resample_replay(b, r, q)       # replay: one device rate + lattice (see 9c/9d)
    b[:, :2] = np.clip(b[:, :2], 0, 200)
    return b

bots = []
for lv in LEVELS:
    src = rng.choice(work_idx, size=int(BOT_SHARE * len(work_idx)), replace=False)
    for i in src:
        bots.append(dict(game=humans[i]["game"], xy=make_bot(humans[i], lv), level=lv, src=int(i)))
print(f"{len(bots)} bot sessions:", dict(Counter(b["level"] for b in bots)))

In [ ]:
# 7. Features for the three views
def dyn(a):
    p, dn = a[:, :2].astype(np.float64), a[:, 2]
    v = np.diff(p, axis=0) / TICK; sp = np.hypot(v[:, 0], v[:, 1])
    acc = np.diff(sp) / TICK if len(sp) > 1 else np.zeros(1)
    jerk = np.diff(acc) / TICK if len(acc) > 1 else np.zeros(1)
    mv = sp > 1.0
    ang = np.arctan2(v[:, 1], v[:, 0]); dang = np.abs((np.diff(ang) + np.pi) % (2 * np.pi) - np.pi)
    dang = dang[mv[1:] & mv[:-1]] if len(dang) else np.zeros(1)
    plen = sp.sum() * TICK; disp = np.hypot(*(p[-1] - p[0]))
    idle = ~mv; runs = np.diff(np.r_[0, idle.astype(int), 0]); n_pause = int((runs == 1).sum())
    presses = int((np.diff(dn) > 0).sum()); dur = len(a) * TICK
    hist = np.histogram(ang[mv], bins=8, range=(-np.pi, np.pi))[0] + 1e-9; hist = hist / hist.sum()
    # local speed minima per second = sub-movements
    s_sm = np.convolve(sp, np.ones(9) / 9, "same")
    minima = int(((s_sm[1:-1] < s_sm[:-2]) & (s_sm[1:-1] < s_sm[2:]) & (s_sm[1:-1] < np.percentile(s_sm, 60))).sum()) if len(s_sm) > 2 else 0
    # tremor band energy share (8–12 Hz) of the speed signal
    if len(sp) > 64:
        F = np.abs(np.fft.rfft(sp - sp.mean())) ** 2; fr = np.fft.rfftfreq(len(sp), TICK)
        trem = F[(fr >= 8) & (fr <= 12)].sum() / (F[fr > 0.5].sum() + 1e-9)
    else: trem = 0.0
    q = lambda x, k: float(np.percentile(x, k)) if len(x) else 0.0
    return [math.log1p(dur), math.log1p(plen), disp / (plen + 1e-9),
            sp.mean(), sp.std(), q(sp, 10), q(sp, 50), q(sp, 90), sp.max(),
            np.abs(acc).mean(), acc.std(), np.abs(jerk).mean(), jerk.std(),
            dang.mean() if len(dang) else 0, dang.std() if len(dang) else 0,
            idle.mean(), n_pause / dur, dn.mean(), presses / dur,
            float(-(hist * np.log(hist)).sum()), minima / dur, trem,
            float((np.abs(np.diff(p, axis=0)).sum(1) == 0).mean()),       # exactly repeated positions
            np.ptp(p[:, 0]), np.ptp(p[:, 1]),
            float(np.corrcoef(sp[:-1], sp[1:])[0, 1]) if len(sp) > 3 and sp.std() > 0 else 0.0]

STAT_NAMES = ["log_dur", "log_path", "straightness", "speed_mean", "speed_std", "speed_p10", "speed_p50",
              "speed_p90", "speed_max", "acc_mean", "acc_std", "jerk_mean", "jerk_std", "turn_mean", "turn_std",
              "idle_share", "pauses_per_s", "down_share", "presses_per_s", "direction_entropy",
              "submovements_per_s", "tremor_share", "repeat_share", "x_range", "y_range", "speed_autocorr"]

def seq_view(a):
    s = a[::SEQ_STRIDE]
    d = np.diff(s[:, :2], axis=0)
    x = np.c_[d / 5.0, s[1:, 2]].astype(np.float32)[:SEQ_MAX]
    return x if len(x) else np.zeros((1, 3), np.float32)

def graph_view(a):
    s = a[::SEQ_STRIDE, :2]; d = np.diff(s, axis=0); mv = np.hypot(d[:, 0], d[:, 1]) > 0.3
    dirs = ((np.arctan2(d[:, 1], d[:, 0]) + np.pi) / (2 * np.pi) * 8).astype(int) % 8
    dm = dirs[mv]; M = np.ones((8, 8))
    for x, y in zip(dm[:-1], dm[1:]): M[x, y] += 1
    M = M / M.sum(1, keepdims=True)
    cells = (np.clip(s[:, 0], 0, 199.9) // 20 * 10 + np.clip(s[:, 1], 0, 199.9) // 20).astype(int)
    ch = cells[np.r_[True, cells[1:] != cells[:-1]]]
    tr = Counter(zip(ch[:-1], ch[1:])); tot = max(sum(tr.values()), 1)
    pt = np.array(list(tr.values())) / tot if tr else np.ones(1)
    seen, rev = set(), 0
    for c in ch: rev += c in seen; seen.add(c)
    return np.r_[M.ravel(), float(-(pt * np.log(pt)).sum()), rev / max(len(ch), 1), len(seen) / 100.0].astype(np.float32)

GRAPH_NAMES = [f"dir{i}->{j}" for i in range(8) for j in range(8)] + ["cell_transition_entropy", "cell_revisit_ratio", "unique_cells"]
ALL = humans + bots
Y = np.array([0] * len(humans) + [1] * len(bots))
LEVEL = np.array(["human"] * len(humans) + [b["level"] for b in bots])
ORIGIN = np.array(list(range(len(humans))) + [b["src"] for b in bots])     # source human of each bot
t0 = time.time()
X_STAT = np.nan_to_num(np.array([dyn(s["xy"]) for s in ALL], dtype=np.float32))
SEQS = [seq_view(s["xy"]) for s in ALL]
X_GRAPH = np.array([graph_view(s["xy"]) for s in ALL], dtype=np.float32)
print(f"features for {len(ALL)} sessions in {time.time() - t0:.0f}s · stat {X_STAT.shape[1]} · graph {X_GRAPH.shape[1]}")

In [ ]:
# 8. Self-supervised next-step encoder, pretrained on the disjoint human set only (no bots, no evaluation humans)
torch.manual_seed(0)
def pad(seqs):
    L = np.array([len(s) for s in seqs]); X = np.zeros((len(seqs), SEQ_MAX, 3), np.float32)
    for i, s in enumerate(seqs): X[i, :len(s)] = s
    return torch.tensor(X), torch.tensor(L)

class NextStep(nn.Module):
    def __init__(self, h=SSL_HID):
        super().__init__(); self.inp = nn.Linear(3, 32); self.rnn = nn.GRU(32, h, num_layers=2, batch_first=True)
        self.out = nn.Linear(h, 3)
    def forward(self, x):
        o, _ = self.rnn(torch.relu(self.inp(x))); return self.out(o), o

def step_loss(m, x, ln):
    """Per-step loss of predicting step t+1 from steps ≤ t: squared error on (dx, dy) + BCE on the button."""
    pred, o = m(x[:, :-1]); tgt = x[:, 1:]
    mask = (torch.arange(x.shape[1] - 1, device=x.device)[None] < (ln.to(x.device)[:, None] - 1)).float()
    l = ((pred[..., :2] - tgt[..., :2]) ** 2).sum(-1) + F.binary_cross_entropy_with_logits(pred[..., 2], tgt[..., 2], reduction="none")
    return l, mask, o

def batches(X, L, idx, bs):
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]; T = int(L[b].max())
        yield X[b, :T].to(DEVICE), L[b]

PX, PL = pad([seq_view(a) for a in pre])
perm = np.random.default_rng(0).permutation(len(PX)); n_va = max(1, len(PX) // 20)
p_va, p_tr = perm[:n_va], perm[n_va:]
ssl = NextStep().to(DEVICE); opt = torch.optim.Adam(ssl.parameters(), lr=SSL_LR)

@torch.no_grad()
def mean_loss(idx):
    ssl.eval(); s = n = 0.0
    for x, ln in batches(PX, PL, idx, 1024):
        l, mask, _ = step_loss(ssl, x, ln); s += float((l * mask).sum()); n += float(mask.sum())
    return s / max(n, 1)

best, state, wait, t0 = 1e9, None, 0, time.time()
for ep in range(SSL_EPOCHS):
    ssl.train(); rs = np.random.default_rng(ep).permutation(p_tr)
    for x, ln in batches(PX, PL, rs, SSL_BATCH):
        l, mask, _ = step_loss(ssl, x, ln); loss = (l * mask).sum() / mask.sum().clamp(min=1)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(ssl.parameters(), 1.0); opt.step()
    v = mean_loss(p_va); print(f"epoch {ep:2d}  val next-step loss {v:.4f}  ({time.time() - t0:.0f}s)")
    if v < best - 1e-4: best, wait, state = v, 0, copy.deepcopy(ssl.state_dict())
    else:
        wait += 1
        if wait >= SSL_PATIENCE: break
ssl.load_state_dict(state)

@torch.no_grad()
def ssl_scores(seqs):
    """Per session: mean next-step prediction error, and the mean GRU state (embedding)."""
    ssl.eval(); X, L = pad(seqs); err, emb = [], []
    for x, ln in batches(X, L, np.arange(len(X)), 1024):
        l, mask, o = step_loss(ssl, x, ln); n = mask.sum(1).clamp(min=1)
        err.append(((l * mask).sum(1) / n).cpu()); emb.append(((o * mask[..., None]).sum(1) / n[:, None]).cpu())
    return torch.cat(err).numpy(), torch.cat(emb).numpy()

SSL_ERR, SSL_EMB = ssl_scores(SEQS)
print("mean next-step error by level:", pd.Series(SSL_ERR).groupby(LEVEL).median().round(4).to_dict())


In [ ]:
# 9. E2 per seed (same splits as v5): supervised RF, human-only detectors, and hybrids
def fpr_threshold(y, s):
    h = np.sort(s[y == 0]); k = int(np.floor(TARGET_FPR * len(h)))
    return float(np.nextafter(h[len(h) - k - 1], np.inf))

def knn_score(ref, X, k=KNN_K):
    nn_ = NearestNeighbors(n_neighbors=k).fit(ref); return nn_.kneighbors(X)[0].mean(1)

HYBRIDS = {"RF | SSL error (primary)": ["RF (stat)", "SSL next-step error"],
           "RF | SSL embedding kNN": ["RF (stat)", "SSL embedding kNN"],
           "RF | kNN (stat)": ["RF (stat)", "kNN (stat)"],
           "RF | GMM (stat)": ["RF (stat)", "GMM (stat)"],
           "RF | kNN | SSL error": ["RF (stat)", "kNN (stat)", "SSL next-step error"]}
rows, t_all = [], time.time()
for seed in SEEDS:
    rest, te_h = train_test_split(work_idx, test_size=0.2, stratify=games_arr[work_idx], random_state=seed)
    tr_h, va_h = train_test_split(rest, test_size=0.2, stratify=games_arr[rest], random_state=seed)
    side = np.full(len(ALL), -1)
    for k, part in enumerate([tr_h, va_h, te_h]): side[np.isin(ORIGIN, part)] = k
    tr_i, va_i, te_i = (np.flatnonzero(side == k) for k in range(3))
    tr_i = tr_i[~np.isin(LEVEL[tr_i], REPLAY)]; va_i = va_i[~np.isin(LEVEL[va_i], REPLAY)]   # E2: replay never seen
    trH = tr_i[Y[tr_i] == 0]                                                                  # training humans only
    yv, yt, lt = Y[va_i], Y[te_i], LEVEL[te_i]
    SC = {}

    def record_pred(name, pred, score, kind):
        row = dict(seed=seed, model=name, kind=kind, ROC_AUC=roc_auc_score(yt, score),
                   FPR=float(pred[yt == 0].mean()), recall_all=float(pred[yt == 1].mean()))
        for lv in LEVELS: row[lv] = float(pred[lt == lv].mean())
        rows.append(row)

    def record(name, s_all, kind):                    # s_all: anomaly / bot score for every session in ALL
        s_va, s_te = s_all[va_i], s_all[te_i]
        record_pred(name, s_te >= fpr_threshold(yv, s_va), s_te, kind); SC[name] = (s_va, s_te)

    def any_view(name, views):
        pv = []
        for v in views:
            s_va, s_te = SC[v]; h = np.sort(s_va[yv == 0])
            pv.append((len(h) - np.searchsorted(h, s_te, side="left") + 1) / (len(h) + 1))
        if (len(yv) - yv.sum() + 1) * TARGET_FPR / len(views) < 1: print(f"⚠️ {name}: too few validation humans for the Bonferroni threshold")
        p_min = np.min(pv, axis=0)
        record_pred(name, p_min <= TARGET_FPR / len(views), -np.log(p_min), "hybrid")

    # supervised baseline (identical to v5 "Random Forest (stat)")
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=seed).fit(X_STAT[tr_i], Y[tr_i])
    record("RF (stat)", rf.predict_proba(X_STAT)[:, 1], "supervised")

    # human-only on the stat features (scaler and models fit on training humans only)
    Z = StandardScaler().fit(X_STAT[trH]).transform(X_STAT)
    record("Isolation Forest (stat)", -IsolationForest(n_estimators=300, random_state=seed).fit(Z[trH]).score_samples(Z), "human-only")
    record("GMM (stat)", -GaussianMixture(n_components=8, covariance_type="full", reg_covar=1e-3, random_state=seed).fit(Z[trH]).score_samples(Z), "human-only")
    record("Mahalanobis (stat)", LedoitWolf().fit(Z[trH]).mahalanobis(Z), "human-only")
    record("kNN (stat)", knn_score(Z[trH], Z), "human-only")

    # human-only SSL (pretrained on disjoint humans; nothing here is fit on bots)
    record("SSL next-step error", SSL_ERR, "human-only")
    E = StandardScaler().fit(SSL_EMB[trH]).transform(SSL_EMB)
    record("SSL embedding kNN", knn_score(E[trH], E), "human-only")

    for name, views in HYBRIDS.items(): any_view(name, views)
    r = pd.DataFrame(rows); last = r[r.seed == seed].set_index("model")
    print(f"seed {seed} ({time.time() - t_all:.0f}s) — replay recall (rotated · raw) and min L1–L4 recall at FPR≤{TARGET_FPR:.0%}:")
    for k in last.index:
        print(f"    {k:<28} {last.loc[k, 'human_replay']:.2f} · {last.loc[k, 'human_replay_raw']:.2f}   L1–L4 min {last.loc[k, LEVELS_SYN].min():.2f}   FPR {last.loc[k, 'FPR']:.3f}")


In [ ]:
# 10. Results and the pre-registered test
res = pd.DataFrame(rows); res["replay_mean"] = res[REPLAY].mean(1)
cols = ["ROC_AUC", "FPR"] + LEVELS + ["replay_mean"]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
summary = res.groupby(["kind", "model"], sort=False)[cols].mean().round(3)
display(summary)
display(res.groupby("model", sort=False)[["replay_mean", "FPR"]].std().round(3).rename(columns=lambda c: c + "_sd"))

def prereg(name, base="RF (stat)"):
    a = res[res.model == name].set_index("seed").replay_mean; b = res[res.model == base].set_index("seed").replay_mean
    wins = int(((a - b) >= 0.10).sum()); syn = res[res.model == name][LEVELS_SYN].mean()
    ok = wins >= 4 and bool((syn >= 0.95).all())
    return dict(model=name, seeds_won=f"{wins}/{len(a)}", mean_gain=round(float((a - b).mean()), 3),
                min_L1_L4=round(float(syn.min()), 3), passed=ok)
pr = pd.DataFrame([prereg(n) for n in HYBRIDS])
print("Pre-registered test (primary = RF | SSL error; the others are exploratory):"); display(pr)

res.to_csv(f"{OUT}/human_only_raw.csv", index=False); summary.to_csv(f"{OUT}/human_only_table.csv"); pr.to_csv(f"{OUT}/prereg.csv", index=False)
fig, ax = plt.subplots(figsize=(15, 5))
m = res.groupby("model", sort=False)[LEVELS].mean()
m.T.plot(kind="bar", ax=ax, width=0.85); ax.set_ylabel(f"recall at FPR ≤ {TARGET_FPR:.0%}"); ax.set_ylim(0, 1.02)
ax.set_title("E2 (replay unseen): supervised RF, human-only detectors and hybrids"); ax.tick_params(axis="x", rotation=15)
ax.legend(fontsize=7, ncol=3, loc="lower left"); plt.tight_layout(); plt.savefig(f"{OUT}/fig_human_only.png", dpi=200); plt.show()

import zipfile
date_files()
with zipfile.ZipFile(f"{BASE}/captcha_human_only_{RUN_DATE}.zip", "w") as z:
    for p in os.listdir(OUT): z.write(f"{OUT}/{p}", p)
print(f"saved {BASE}/captcha_human_only_{RUN_DATE}.zip")
